# Equity duration: a reproducible data tour

This notebook reads cleaned pipeline artifacts, explains their units and checks, and traces the agreed Sector ER Tracker analysis. It does not display individual licensed stock records.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.cruft.json').exists())
sys.path.insert(0, str(root / 'src'))
import pandas as pd
from p22.common import CLEAN, OUT
from p22.sector.data import RESULTS
from p22.data_quality import QUALITY
history = pd.read_parquet(CLEAN / 'us_updated.parquet')
pd.DataFrame({'first_year':[int(history.Year.min())], 'last_year':[int(history.Year.max())], 'observations':[len(history)], 'duplicate_years':[int(history.Year.duplicated().sum())]})

,first_year,last_year,observations,duplicate_years
0,1871,2025,155,0


## Check the data before interpreting the estimates

Independent cash events and return fields use different reinvestment conventions. The pipeline checks their agreement, issuer links, portfolio weights and coverage. A check is not passed merely because an estimate looks like the paper.

In [2]:
pd.read_csv(QUALITY / 'checks.csv')

,check,observed,tolerance,passed,meaning
0,Inflation reconciles with CPI,0.000000e+00,1.000000e-10,True,Independent CPI and reported inflation agree.
1,Author tercile counts sum to all,0.000000e+00,0.000000e+00,True,All eligible firms are allocated once.
2,Clean stock-month duplicate keys,0.000000e+00,0.000000e+00,True,A security cannot receive duplicate return wei...
3,Cash event matching failure fraction,2.203685e-04,1.000000e-03,True,Ex-date factors and preceding prices are avail...
4,Portfolio weights sum to one,6.661338e-16,1.000000e-10,True,Weights conserve invested portfolio value.
5,Monthly cash return identity,2.775558e-17,1.000000e-10,True,Unit price change and withdrawn cash reconcile...
6,Cash versus effective yield: paying months,1.745259e-04,1.000000e-03,True,Small differences are expected from ex-date re...


## Paper replication versus updated inputs

Tables 6 and 7 use the authors' prepared portfolios for replication. The updated stock-level construction has a shorter available history. Its overlap diagnostics are reported rather than concealed.

In [3]:
pd.read_csv(OUT / 'update/cross_source_checks.csv')

,portfolio,years,first_year,last_year,dp_correlation,return_correlation,growth_correlation,missing_rebuilt_earnings_years,mean_firm_count_ratio
0,All,53,1970,2022,0.996423,0.998521,0.870243,0,0.989479
1,Low,53,1970,2022,0.977753,0.992452,0.308901,3,0.995376
2,Medium,53,1970,2022,0.989366,0.993122,0.663039,0,0.984845
3,High,53,1970,2022,0.986913,0.994152,0.717097,3,0.988236
4,Below 0.5,53,1970,2022,0.992701,0.997089,0.769115,0,0.998311
5,Above 0.5,53,1970,2022,0.988356,0.994400,0.521703,4,0.976893


## Sector ER Tracker

Established dividend payers are sorted within industries. Both sides have identical monthly industry weights. ER measures a model-dependent valuation decomposition, not an expected return. Wide intervals or a null result are valid findings.

In [4]:
pd.read_csv(RESULTS / 'comparison.csv')

,industry_id,comparison,n_obs,start,end,low_ER,high_ER,low_minus_high_ER,lower,upper
0,0,Industry-neutral,31,1973,2025,0.513164,0.395775,0.117389,-1.566530,3.816229
1,-1,Market-wide,31,1973,2025,0.175857,0.430571,-0.254714,-10.905313,6.385229


## Follow the calculation

The same consecutive annual pairs feed return, dividend-growth and dividend-price regressions. The next cell independently reruns the scalar estimator on one cleaned aggregate portfolio. The separate VAR sensitivity adds dividend growth as a forecasting state.

In [5]:
from p22.sector.estimate import pairs, fit
from p22.sector.data import config
annual = pd.read_csv(RESULTS / 'annual.csv')
market = annual[(annual.industry_id == -2) & (annual.portfolio == 'All')]
pd.Series(fit(pairs(market), config()))

n_obs                                                                         55
beta_return                                                             0.059991
beta_growth                                                            -0.051646
beta_dp                                                                  0.91523
rho                                                                     0.972512
denominator                                                             0.109928
ER                                                                       0.54573
EDG                                                                     0.469817
coefficient_restriction                                                 1.001709
window_start                                                                1971
window_end                                                                  2025
bootstrap_valid_fraction                                                     1.0
bootstrap_unstable_fraction 

## Rebuild and trace sources

Run `uv run doit` with your own WRDS credentials to regenerate the complete submission. `uv run doit replicate checks` runs the public replication and tests without WRDS. Source descriptions and links are registered in ChartBook; data and output directories are machine settings, while analysis dates are committed.